# Model Context Protocol (MCP)

MCP standardizes how LLMs discover and call external tools — think of it as a USB-C port for AI models. An MCP server exposes tools via a JSON schema; clients (agents) discover and invoke them. The result is modular, reusable tooling that works across frameworks.

## Implementation with Flyte v2

This notebook reimplements the ADK `MCPToolset` + FastMCP server from Chapter 10 using **Flyte v2 primitives**.

#### ADK/FastMCP vs Flyte v2 — Key Differences

| Aspect | ADK / FastMCP | Flyte v2 |
|--------|---------------|----------|
| **Tool definition** | `@mcp_server.tool` decorator | `@env.task` — tasks are typed, discoverable tools |
| **Tool schema** | JSON schema auto-generated from type hints + docstring | Python type hints are the schema — no conversion needed |
| **Server transport** | HTTP / SSE / stdio (npx, uvx) | Flyte's gRPC task protocol |
| **Tool discovery** | Agent Card JSON at `/.well-known/agent.json` | `flyte.remote.Task.get("team.task_name")` |
| **Orchestration** | `LlmAgent(tools=[MCPToolset(...)])` | Agent task calls tool tasks directly via `await` |
| **Secrets** | `os.environ` at import time | `flyte.Secret` injected at task execution time |
| **Execution** | In-process or subprocess | Container on Kubernetes |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' anthropic fastmcp

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-ant-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from datetime import timedelta

import anthropic
import flyte

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="mcp-agent", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.40.0", "fastmcp>=2.0.0")
)

mcp_env = flyte.TaskEnvironment(
    name="mcp_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define MCP tools as Flyte tasks

In the ADK example, tools are registered on an MCP server with `@mcp_server.tool` — a decorator that wraps a Python function and auto-generates a JSON schema from its type hints and docstring. Agents discover them via an Agent Card.

In Flyte v2, `@env.task` serves the same purpose: it wraps a function with typed I/O, makes it discoverable via `flyte.remote.Task.get()`, and runs it in a container. No separate server process needed.

In [ ]:
@dataclass
class ToolResult:
    """Output from an MCP-style tool call."""
    tool_name: str
    success: bool
    output: str
    error: str = ""


@dataclass
class AgentResponse:
    """Final response from the MCP agent."""
    user_query: str
    tools_called: list[str]
    final_answer: str


# ── Tool 1: greet ────────────────────────────────────────────────────────────
# Equivalent to the FastMCP example: @mcp_server.tool def greet(name: str) -> str

@mcp_env.task(cache="auto")
async def greet(name: str) -> ToolResult:
    """Generates a personalized greeting."""
    return ToolResult(
        tool_name="greet",
        success=True,
        output=f"Hello, {name}! Nice to meet you.",
    )


# ── Tool 2: calculate ─────────────────────────────────────────────────────────

@mcp_env.task(cache="auto")
async def calculate(expression: str) -> ToolResult:
    """Safely evaluates a simple arithmetic expression (e.g. '2 + 2 * 3')."""
    import ast
    try:
        # Allow only safe literal math
        tree = ast.parse(expression, mode="eval")
        for node in ast.walk(tree):
            if not isinstance(node, (ast.Expression, ast.BinOp, ast.UnaryOp,
                                     ast.Num, ast.Constant,
                                     ast.Add, ast.Sub, ast.Mult, ast.Div,
                                     ast.Pow, ast.Mod, ast.FloorDiv,
                                     ast.USub, ast.UAdd)):
                raise ValueError(f"Unsafe node: {type(node).__name__}")
        result = eval(compile(tree, "<expr>", "eval"))  # noqa: S307
        return ToolResult(tool_name="calculate", success=True, output=str(result))
    except Exception as e:
        return ToolResult(tool_name="calculate", success=False, output="", error=str(e))


# ── Tool 3: word_count ────────────────────────────────────────────────────────

@mcp_env.task(cache="auto")
async def word_count(text: str) -> ToolResult:
    """Counts the number of words in a text string."""
    count = len(text.split())
    return ToolResult(
        tool_name="word_count",
        success=True,
        output=f"{count} words",
    )

### 5. Define the Anthropic tool schemas

In the ADK / FastMCP pattern, an LLM agent receives tool descriptions via the MCP protocol and selects the right tool. Here we expose the same three tools to Claude using Anthropic's `tool_use` format — the JSON schema mirrors what FastMCP would auto-generate from the function signatures above.

In [ ]:
# Anthropic tool schema — equivalent to FastMCP's auto-generated schema
MCP_TOOLS = [
    {
        "name": "greet",
        "description": "Generates a personalized greeting for a given name.",
        "input_schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "The name of the person to greet."}
            },
            "required": ["name"],
        },
    },
    {
        "name": "calculate",
        "description": "Evaluates a simple arithmetic expression (e.g. '2 + 2 * 3').",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Arithmetic expression to evaluate."}
            },
            "required": ["expression"],
        },
    },
    {
        "name": "word_count",
        "description": "Counts the number of words in a text string.",
        "input_schema": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "Text to count words in."}
            },
            "required": ["text"],
        },
    },
]

AGENT_SYSTEM = """\
You are a helpful assistant with access to three tools: greet, calculate, and word_count.
Use them to answer the user's request. After receiving tool results, compose a final answer."""

### 6. Define the MCP agent task

In the ADK pattern, `LlmAgent(tools=[MCPToolset(...)])` handles tool discovery and invocation transparently. Here the same loop is explicit:
1. Call Claude with tool schemas → receive `tool_use` blocks
2. Dispatch to the matching Flyte task (equivalent to the MCP server routing the call)
3. Feed results back → repeat until `end_turn`

In [ ]:
async def _dispatch_tool(name: str, inputs: dict) -> str:
    """Route a tool_use block to the appropriate Flyte task."""
    if name == "greet":
        result = await greet(name=inputs["name"])
    elif name == "calculate":
        result = await calculate(expression=inputs["expression"])
    elif name == "word_count":
        result = await word_count(text=inputs["text"])
    else:
        return f"Unknown tool: {name}"
    return result.output if result.success else f"Error: {result.error}"


@mcp_env.task(
    retries=2,
    timeout=timedelta(minutes=3),
    cache=flyte.Cache(behavior="disable"),
)
async def mcp_agent(user_query: str, max_steps: int = 8) -> AgentResponse:
    """
    MCP-style agent: calls Flyte tasks as tools via Anthropic tool_use.

    Replaces ADK's:
      root_agent = LlmAgent(
          model="gemini-2.0-flash",
          tools=[MCPToolset(connection_params=StdioServerParameters(...))],
      )

    Flyte tasks play the role of MCP server tools — same typed interface,
    no separate server process required.
    """
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    messages = [{"role": "user", "content": user_query}]
    tools_called: list[str] = []
    final_answer = ""

    for _ in range(max_steps):
        response = await client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=512,
            system=AGENT_SYSTEM,
            tools=MCP_TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    final_answer = block.text
            break

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    tools_called.append(block.name)
                    result_text = await _dispatch_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result_text,
                    })
            messages.append({"role": "user", "content": tool_results})

    return AgentResponse(
        user_query=user_query,
        tools_called=tools_called,
        final_answer=final_answer,
    )

### 7. Run locally

In [ ]:
QUERIES = [
    "Say hello to Alice and also calculate 15 * 7 + 3.",
    "How many words are in: 'The quick brown fox jumps over the lazy dog'?",
    "Greet Bob, then tell me what 100 divided by 4 is.",
]

for query in QUERIES:
    run = flyte.run(mcp_agent, user_query=query)
    run.wait()
    result: AgentResponse = run.outputs()[0]
    print(f"Query: {result.user_query}")
    print(f"Tools: {result.tools_called}")
    print(f"Answer: {result.final_answer}")
    print("-" * 60)

### FastMCP server (standalone)

To expose your tools to external MCP clients (other frameworks, Claude Desktop), you can also run a `fastmcp` server alongside your Flyte tasks. The tools are identical — just registered on both the MCP server and as Flyte tasks.

In [ ]:
# fastmcp_server.py — run separately with: python fastmcp_server.py
# This is the FastMCP equivalent of the ADK MCPToolset example.

from fastmcp import FastMCP

mcp_server = FastMCP(name="flyte-tools")


@mcp_server.tool
def greet_mcp(name: str) -> str:
    """Generates a personalized greeting."""
    return f"Hello, {name}! Nice to meet you."


@mcp_server.tool
def calculate_mcp(expression: str) -> str:
    """Evaluates a simple arithmetic expression."""
    import ast
    tree = ast.parse(expression, mode="eval")
    return str(eval(compile(tree, "<expr>", "eval")))  # noqa: S307


@mcp_server.tool
def word_count_mcp(text: str) -> str:
    """Counts the number of words in text."""
    return f"{len(text.split())} words"


# To run the server (HTTP on port 8000):
# mcp_server.run(transport="http", host="0.0.0.0", port=8000)

print("FastMCP server defined — tools:", [t for t in ["greet_mcp", "calculate_mcp", "word_count_mcp"]])

### Running remotely

With `ReusePolicy`, the MCP agent pods stay warm between calls — critical for interactive tool-use loops where cold-starting a container per query would dominate latency. The `AgentResponse` dataclass makes tool call traces fully auditable in the Flyte UI.

> **Note:** `ReusePolicy` is a Union-specific feature that requires a [Union deployment](https://www.union.ai/docs/v2/union/). It is not supported on the local devbox.

In [ ]:
run = flyte.run(mcp_agent, user_query="Greet Claude, calculate 42 * 8, and count the words in 'Hello World'.")
run.wait()
result = run.outputs()[0]
print(f"Tools used: {result.tools_called}")
print(result.final_answer)